# Bulk library build (continuation + exploit) — long GPU run

Builds a large, texture-diverse **continuation** + **exploit** library across many boards, on
the correct streets=3 batched GPU solver. Sized for a **~10–20h** session and **checkpointed** —
each generator re-writes a valid, signed pack every `CHECKPOINT` boards, so if the Kaggle
session hits its time limit the latest pack on disk is still complete and downloadable.

**Sizing** (reference: 4 boards @ n=80, iters=600 ≈ 8–9 min ≈ ~2 min/board; time scales roughly
as `boards × iters × n²`). Defaults below target ~15h across both generators. Cut `BOARDS` or
`N`/`ITERS` to shorten. Boards come from a deterministic diverse set (max 180; `_DIVERSE_FLOPS`
× 6 runouts in demo/gen_continuation.py).

Pick a **GPU** session; **Internet On**. Writes new versions `continuation_full` / `exploit_full`
(does NOT overwrite the shipped `continuation_seed` / `exploit_v1`).

In [ ]:
# Fail fast if this isn't a working GPU session.
import cupy, subprocess
ndev = cupy.cuda.runtime.getDeviceCount()
assert ndev > 0, 'No CUDA device — set the runtime Accelerator to GPU.'
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
print(f'CuPy sees {ndev} GPU(s) — good to go')

In [ ]:
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
import subprocess
print('source ready @', subprocess.run(['git','-C','/kaggle/working/poker','rev-parse','--short','HEAD'],
                                       capture_output=True, text=True).stdout.strip())

In [ ]:
# --- knobs (target ~15h across BOTH generators; each ~= half) ---
#   BOARDS: diverse boards per generator (max 180).  N: combos/range.  ITERS: CFR iterations.
#   CHECKPOINT: re-write the signed pack every K boards (session-safe). Progress prints
#   `[i/total] ... stable=True` per board and `wrote ... pack` at each checkpoint.
import subprocess, os
BOARDS, N, ITERS, CHECKPOINT = 64, 120, 800, 4
env = {**os.environ, 'PYTHONPATH': 'src'}
def gen(script, version):
    subprocess.run(['python', script, '--solver', 'gpu', '--dtype', 'float32',
                    '--boards', str(BOARDS), '--n', str(N), '--iters', str(ITERS),
                    '--checkpoint-every', str(CHECKPOINT), '--version', version],
                   cwd='/kaggle/working/poker', env=env, check=True)
gen('demo/gen_continuation.py', 'continuation_full')
gen('demo/gen_exploit.py', 'exploit_full')

In [ ]:
# Expose whatever packs exist for download (works even if the session was cut off mid-run —
# the last checkpoint of each version is a valid signed pack).
import shutil, os, glob
base = '/kaggle/working/poker/output/packs'
got = []
for version in ('continuation_full', 'exploit_full'):
    for f in [f'flop_pack_{version}.db', f'flop_pack_{version}.db.gz', f'build_report_{version}.json']:
        p = os.path.join(base, f)
        if os.path.exists(p):
            shutil.copy(p, os.path.join('/kaggle/working', f)); got.append(f)
print('DOWNLOAD from /kaggle/working:' if got else 'No packs written yet — check the solve cell.')
for f in got:
    print('  %-42s %d KB' % (f, os.path.getsize(os.path.join('/kaggle/working', f)) // 1024))